In [8]:
import pandas as pd

# Load data
poi   = pd.read_csv("../data/processed/poi_grid_malang.csv")
crowd = pd.read_csv("../data/processed/crowd_per_grid.csv")

In [9]:
# Separate tier 4 from crowd
crowd_valid = crowd[crowd["tier"] != 4].copy()
crowd_tier4 = crowd[crowd["tier"] == 4][["grid_id", "tier"]].copy()

# Get crowd metadata per grid (primary tier & source_venue)
# primary source_venue = venue most often the source of MAX crowd
source_meta = (
    crowd_valid
    .groupby("grid_id")
    .agg(
        tier                 = ("tier", "min"),
        primary_source_venue = ("source_venue", lambda x: x.value_counts().idxmax())
    )
    .reset_index()
)

In [10]:

crowd_valid["day_int"] = crowd_valid["day_int"].astype(int) 
crowd_valid["hour"]    = crowd_valid["hour"].astype(int)

# Create crowd column labels: crowd_mon_06, crowd_tue_14, etc.
DAY_MAP = {0: "mon", 1: "tue", 2: "wed", 3: "thu", 4: "fri", 5: "sat", 6: "sun"}
crowd_valid["col_label"] = (
    "crowd_"
    + crowd_valid["day_int"].map(DAY_MAP)
    + "_"
    + crowd_valid["hour"].astype(str).str.zfill(2)
)

# Pivot crowd: long → wide
crowd_wide = crowd_valid.pivot_table(
    index   = "grid_id",
    columns = "col_label",
    values  = "crowd_score",
    aggfunc = "max"      
).reset_index()

# Sort crowd columns to be neat (mon_00 → sun_23)
day_order  = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]
hour_order = [str(h).zfill(2) for h in range(24)]
col_order  = [
    f"crowd_{d}_{h}"
    for d in day_order
    for h in hour_order
    if f"crowd_{d}_{h}" in crowd_wide.columns
]
crowd_wide = crowd_wide[["grid_id"] + col_order]

In [11]:
# Join all components
# 1. POI (base)
master = poi.copy()

# 2. Crowd metadata (tier & primary source_venue) for non-tier-4
master = master.merge(source_meta, on="grid_id", how="left")

# 3. Tier 4 — fill remaining NaN tiers
tier4_map = crowd_tier4.set_index("grid_id")["tier"]
master["tier"] = master["tier"].fillna(master["grid_id"].map(tier4_map)).astype(int)

# 4. Crowd wide
master = master.merge(crowd_wide, on="grid_id", how="left")

In [12]:
# Arrange final column order
base_cols = [
    "grid_id", "lat", "lon",
    "n_campus", "n_school", "n_market",
    "n_place_of_worship", "n_park", "n_residential",
    "dominant_zone",
    "tier", "primary_source_venue"
]
master = master[base_cols + col_order]

# Save
master.to_csv("../data/processed/master_features_malang.csv", index=False)
